# 02 — Análisis Longitudinal

**Observatorio de Ciencia, Tecnología e Innovación — Grupo 7**

Este notebook analiza la evolución temporal de los investigadores reconocidos
por Minciencias a través de las **6 convocatorias (2013–2021)**.

**Contenido:**
1. Carga y preparación del dataset consolidado
2. Evolución del número total de investigadores por convocatoria
3. Tendencias por género, área y región
4. Análisis de movilidad en categoría de clasificación
5. Verificación de unicidad de identificadores (ID_PERSONA_PR)
6. Conclusiones longitudinales

In [ ]:
import sys
import pathlib

ROOT = pathlib.Path().resolve().parent  # raiz del repo (notebooks/../)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ingesta import cargar_consolidado
from Transformacion import transformar

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 5)

## 1. Carga de datos

In [ ]:
df_raw = cargar_consolidado()
df = transformar(df_raw)
print(f'Shape: {df.shape}')
df.head()

## 2. Evolución total de investigadores por convocatoria

In [ ]:
evolucion = (
    df.groupby('ANO_CONVO_INT')
    .size()
    .reset_index(name='Investigadores')
)
print(evolucion)

plt.figure(figsize=(8, 5))
plt.bar(
    evolucion['ANO_CONVO_INT'].astype(str),
    evolucion['Investigadores'],
    color='steelblue',
    edgecolor='white',
    width=0.5
)
for _, row in evolucion.iterrows():
    anio, val = row['ANO_CONVO_INT'], row['Investigadores']
    plt.text(str(anio), val + 100, f'{val:,}', ha='center', fontsize=11)
plt.title('Investigadores reconocidos por convocatoria')
plt.xlabel('Año de convocatoria')
plt.ylabel('Número de investigadores')
plt.tight_layout()
plt.show()

## 3. Tendencias por género, área y región

In [ ]:
# Evolución por género
genero_anio = (
    df.groupby(['ANO_CONVO_INT', 'NME_GENERO_PR'])
    .size()
    .reset_index(name='Cantidad')
)

plt.figure(figsize=(10, 5))
for genero, grupo in genero_anio.groupby('NME_GENERO_PR'):
    plt.plot(
        grupo['ANO_CONVO_INT'].astype(str),
        grupo['Cantidad'],
        marker='o',
        label=genero
    )
plt.title('Evolución de investigadores por género')
plt.xlabel('Año de convocatoria')
plt.ylabel('Número de investigadores')
plt.legend(title='Género')
plt.tight_layout()
plt.show()

In [ ]:
# Evolución por gran área (top 5)
top_areas = df['NME_GRAN_AREA_PR'].value_counts().head(5).index.tolist()

area_anio = (
    df[df['NME_GRAN_AREA_PR'].isin(top_areas)]
    .groupby(['ANO_CONVO_INT', 'NME_GRAN_AREA_PR'])
    .size()
    .reset_index(name='Cantidad')
)

plt.figure(figsize=(12, 5))
for area, grupo in area_anio.groupby('NME_GRAN_AREA_PR'):
    plt.plot(
        grupo['ANO_CONVO_INT'].astype(str),
        grupo['Cantidad'],
        marker='o',
        label=area
    )
plt.title('Evolución por gran área de conocimiento (top 5)')
plt.xlabel('Año de convocatoria')
plt.ylabel('Número de investigadores')
plt.legend(title='Gran área', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Movilidad en categoría de clasificación

In [ ]:
clas_anio = (
    df.groupby(['ANO_CONVO_INT', 'NME_CLASIFICACION_PR'])
    .size()
    .unstack(fill_value=0)
)
print('Distribución de categorías por convocatoria:')
print(clas_anio)

clas_anio.T.plot(kind='bar', figsize=(12, 5), colormap='tab10')
plt.title('Categorías de clasificación por convocatoria')
plt.xlabel('Categoría')
plt.ylabel('Número de investigadores')
plt.xticks(rotation=35, ha='right')
plt.legend(title='Convocatoria')
plt.tight_layout()
plt.show()

## 5. Verificación de unicidad de ID_PERSONA_PR

In [ ]:
if 'ID_PERSONA_PR' in df.columns:
    # Investigadores que aparecen en más de una convocatoria
    presencia = (
        df.groupby('ID_PERSONA_PR')['ANO_CONVO_INT']
        .nunique()
        .rename('n_convocatorias')
    )
    print('Distribución de investigadores por número de convocatorias:')
    print(presencia.value_counts().sort_index())

    # Verificación de consistencia de edad
    # Para un investigador entre 2017 y 2019 la diferencia debería ser ~2 años
    df_multi = df[df['ID_PERSONA_PR'].isin(presencia[presencia > 1].index)]
    edad_delta = (
        df_multi.groupby('ID_PERSONA_PR')
        .apply(lambda g: g.sort_values('ANO_CONVO_INT')['EDAD_ANOS_PR'].diff().abs().max())
        .dropna()
    )
    print('\nEstadísticas de la diferencia de edad entre convocatorias:')
    print(edad_delta.describe())
    print(f'IDs con diferencia de edad > 6 años: {(edad_delta > 6).sum():,}')
else:
    print('Columna ID_PERSONA_PR no disponible en el dataset.')

## 6. Conclusiones longitudinales

In [ ]:
print('=== RESUMEN LONGITUDINAL ===')
for anio, g in df.groupby('ANO_CONVO_INT'):
    print(f'\nConvocatoria {int(anio)}:')
    print(f'  Registros       : {len(g):,}')
    if 'ID_PERSONA_PR' in g.columns:
        print(f'  IDs únicos      : {g["ID_PERSONA_PR"].nunique():,}')
    if 'NME_GENERO_PR' in g.columns:
        pct_fem = (g['NME_GENERO_PR'] == 'FEMENINO').mean() * 100
        print(f'  % Femenino      : {pct_fem:.1f}%')
    if 'EDAD_ANOS_PR' in g.columns:
        print(f'  Edad media      : {g["EDAD_ANOS_PR"].mean():.1f} años')